In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

PREDICTORS = [
    "RH_Avg", "RH_Max", "RH_Min",
    "AT_Avg", "AT_Max", "AT_Min",
    "SR_Avg", "SR_Max", "SR_Min",
    "RF_Avg", "RF_Max", "RF_Min",
]
TARGET = "Flood_Risk_Status"

df = pd.read_csv("sylhet_climate_2013_2024.csv")
df = df.drop(columns=[c for c in ["Year", "Month"] if c in df.columns])

for col in PREDICTORS:
    monthly = df.groupby(df.index // 12)[col].transform("mean")
    df[col] = df[col].fillna(monthly)

imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=100, random_state=42), random_state=42)
df[PREDICTORS] = imp.fit_transform(df[PREDICTORS])

rf_cols = [c for c in PREDICTORS if c.startswith("RF")]
rh_cols = [c for c in PREDICTORS if c.startswith("RH")]
df.loc[df[rf_cols].lt(0).any(axis=1), rf_cols] = 0
df.loc[df[rh_cols].gt(100).any(axis=1), rh_cols] = 100

df = df.drop_duplicates().reset_index(drop=True)

scaler = StandardScaler()
df[PREDICTORS] = scaler.fit_transform(df[PREDICTORS])

df.to_csv("flood_processed.csv", index=False)
print(df.shape)
print(df[TARGET].value_counts())


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("flood_processed.csv")
PREDICTORS = [c for c in df.columns if c != "Flood_Risk_Status"]

le = LabelEncoder()
y = le.fit_transform(df["Flood_Risk_Status"])
X = df[PREDICTORS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42),
}

rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1": f1_score(y_test, pred, average="macro", zero_division=0),
    })

results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
results.to_csv("flood_model_results.csv", index=False)
results


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

clf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
classes = le.classes_

print(classification_report(y_test, pred, target_names=classes, digits=4))

cm = confusion_matrix(y_test, pred, normalize="true")
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=".3f", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("flood_confusion_matrix.png", dpi=150)

import shap
explainer = shap.TreeExplainer(clf)
sv = explainer.shap_values(X_train)
plt.figure()
shap.summary_plot(sv, X_train, plot_type="dot", show=False)
plt.tight_layout()
plt.savefig("flood_shap.png", dpi=150)
